In [2]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
import re
from IPython.display import display

# 1. Setup the Plotting Function
def plot_graph(equation, x_range, y_range):
    plt.figure(1, figsize=(8, 5))
    plt.clf()
    
    # Pre-processing for user-friendly syntax
    clean_eq = equation.replace('^', '**')
    # This regex turns '3x' into '3*x' and '2(x)' into '2*(x)'
    clean_eq = re.sub(r'(\d)(?=[a-zA-Z\(])', r'\1*', clean_eq)
    
    try:
        x = np.linspace(x_range[0], x_range[1], 1000)
        
        safe_dict = {
            "x": x, "np": np, "pi": np.pi,
            "sin": np.sin, "cos": np.cos, "tan": np.tan,
            "arcsin": np.arcsin, "arccos": np.arccos, "arctan": np.arctan,
            "log": np.log, "log10": np.log10, "sqrt": np.sqrt,
            "exp": np.exp, "abs": np.abs
        }
        
        # Evaluate the equation
        y = eval(clean_eq, {"__builtins__": None}, safe_dict)
        
        # Handle constant numbers (like y = 2)
        if np.isscalar(y):
            y = np.full_like(x, y)
        
        plt.plot(x, y, label=f"y = {equation}")
        plt.axhline(0, color='black', linewidth=1)
        plt.axvline(0, color='black', linewidth=1)
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.ylim(y_range[0], y_range[1])
        plt.xlim(x_range[0], x_range[1]) # Ensures X matches slider bounds
        plt.title("Interactive Graphing Calculator")
        plt.legend()
        plt.show()
    except Exception as e:
        print(f"Error: {e}")

# 2. Create UI Widgets
eq_input = widgets.Text(
    value='x^2 + 3x - 2',
    description='y =',
    layout=widgets.Layout(width='70%')
)

DEFAULT_BOUNDS = [-10, 10]

x_slider = widgets.FloatRangeSlider(value=DEFAULT_BOUNDS, min=-100, max=100, step=0.1, description='X Zoom:')
y_slider = widgets.FloatRangeSlider(value=DEFAULT_BOUNDS, min=-100, max=100, step=0.1, description='Y Zoom:')

reset_button = widgets.Button(description='Reset View', button_style='info', icon='home')

def on_reset_clicked(b):
    x_slider.value = DEFAULT_BOUNDS
    y_slider.value = DEFAULT_BOUNDS

reset_button.on_click(on_reset_clicked)

# 3. Layout and Logic
out = widgets.interactive_output(plot_graph, {'equation': eq_input, 'x_range': x_slider, 'y_range': y_slider})
header = widgets.HBox([eq_input, reset_button])
ui = widgets.VBox([header, x_slider, y_slider, out])

display(ui)